# Recorte manual de imagens (ROIs)

Notebook de apoio ao `recorte_manual.py`: abre a janela de recorte, aplica ROIs em lote e confere os recortes gerados.

## Fluxos de trabalho

| Situacao | Como fazer |
|---|---|
| Cada foto tem a amostra em um lugar diferente | **Selecao livre**: arraste o retangulo em cada imagem e aperte `Enter` |
| Quer todos os recortes do mesmo tamanho | Marque **Tamanho fixo**, informe `larg x alt` e clique no centro da regiao |
| As fotos foram tiradas na mesma posicao (tripe/scanner) | Marque **Travar ROIs**, desenhe uma vez e aperte `Enter` em cada imagem, ou use **Aplicar estas ROIs a TODAS as imagens** |
| Quer varias sub-imagens por regiao (como no artigo) | Defina a **grade** `linhas x colunas` dentro da ROI |

Os recortes saem sempre na **resolucao original** (o zoom e so visual) e as coordenadas ficam em `recortes_coords.csv`, o que permite refazer tudo depois.

**Atalhos:** `Enter` salva e avanca | setas trocam de imagem | `Del` remove a ultima ROI | `C` limpa | `T` trava/destrava | `+`/`-`/`0` zoom (ou `Ctrl`+roda do mouse).

In [ ]:
from pathlib import Path

import recorte_manual as rec

# ==== ajuste aqui ====
PASTA_IMAGENS = Path(r'imagens/original')          # pasta com as fotos originais
PASTA_SAIDA = Path(r'imagens/recorte')           # onde os recortes serao gravados
FORMATO = 'jpg'                          # png preserva a cor sem perda (melhor p/ histogramas)

PASTA_IMAGENS.mkdir(exist_ok=True)
imagens = rec.listar_imagens(PASTA_IMAGENS)
print(f'{len(imagens)} imagens em {PASTA_IMAGENS.resolve()}')
for caminho in imagens[:10]:
    print(' -', caminho.name)

## 1. Abrir a janela de recorte

A celula abaixo abre a janela do Tkinter. **O notebook fica bloqueado enquanto a janela estiver aberta** - feche a janela para o kernel voltar a responder.

Se preferir trabalhar fora do Jupyter (mais estavel), rode no terminal:

```powershell
.venv\Scripts\python.exe recorte_manual.py --entrada imagens --saida recortes
```

In [ ]:
rec.abrir_janela(PASTA_IMAGENS, PASTA_SAIDA, FORMATO)

## 2. Regiao fixa aplicada a pasta inteira (sem janela)

Quando a ROI ja e conhecida (por exemplo, copiada da lista de ROIs da janela ou do CSV), da para recortar tudo de uma vez. Cada ROI e `(x, y, largura, altura)` em pixels da imagem original.

In [ ]:
ROIS = [
    (300, 200, 600, 400),   # (x, y, largura, altura)
    # (1000, 200, 600, 400),
]
N_LINHAS, N_COLUNAS = 2, 3   # cada ROI vira 2x3 = 6 sub-recortes

total = rec.aplicar_em_lote(PASTA_IMAGENS, PASTA_SAIDA, ROIS,
                            n_linhas=N_LINHAS, n_colunas=N_COLUNAS,
                            formato=FORMATO, subpasta_por_imagem=False)

## 3. Conferir os recortes gerados

Le o `recortes_coords.csv`, resume quantos recortes sairam por imagem e mostra um mosaico de amostra. Tamanhos diferentes entre recortes atrapalham a comparacao de histogramas, por isso vale checar a coluna de dimensoes.

In [ ]:
import pandas as pd

df = pd.read_csv(PASTA_SAIDA / rec.NOME_CSV)
print(f'{len(df)} recortes registrados')
display(df.groupby('imagem').agg(n_recortes=('arquivo_saida', 'count'),
                                 largura=('largura', 'unique'),
                                 altura=('altura', 'unique')))
df.tail()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

amostra = df['arquivo_saida'].tolist()[:12]
n_col = 4
n_lin = max(1, -(-len(amostra) // n_col))
fig, eixos = plt.subplots(n_lin, n_col, figsize=(3 * n_col, 3 * n_lin))
for eixo, nome in zip(fig.axes, amostra):
    eixo.imshow(Image.open(PASTA_SAIDA / nome))
    eixo.set_title(nome, fontsize=7)
for eixo in fig.axes:
    eixo.axis('off')
plt.tight_layout()
plt.show()

## 4. Refazer os recortes a partir do CSV

Reprodutibilidade: com o CSV guardado junto do artigo, qualquer pessoa reproduz exatamente os mesmos recortes (ou os regera em outro formato/pasta) sem repetir a selecao manual.

In [ ]:
rec.refazer_do_csv(PASTA_SAIDA / rec.NOME_CSV, PASTA_IMAGENS,
                   pasta_saida=Path('recortes_refeitos'), formato='png')